In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats

In [ ]:
def get_statistics(scores_a, scores_b, margin):
    differences = scores_a - scores_b

    stat, p_value_noninferior = stats.wilcoxon(scores_b + margin, scores_a, alternative='greater')

    mean_diff = np.mean(differences)
    std_diff = np.std(differences, ddof=1)
    cohens_d = mean_diff / std_diff

    correlation, p_value_correlation = stats.pearsonr(scores_a, scores_b, alternative='greater')

    return correlation, p_value_correlation, p_value_noninferior, cohens_d

In [ ]:
metric = 'VUS-PR' # Use 'VUS-PR' for reproducing results in Table 8, 'Event-based-F1' for reproducing results in Table 9.

model_sizes = ('base', 'large')
win_sizes = (64, 256, 512, 1024)

# The non-inferiority margins for each combination of model size and metric, as mentioned in Appendix J of the paper.
margins = {
    ('base', 'VUS-PR'): 0.0008,
    ('base', 'Event-based-F1'): 0.0001,
    ('large', 'VUS-PR'): 0.016,
    ('large', 'Event-based-F1'): 0.025}

# Initialize an empty DataFrame with the specified column names and data types

column_specs = {
    'Variant': 'object',
    'mean (TSFM)': 'float64',  
    'mean (baseline)': 'float64',      
    'correlation': 'float64',  
    'p-value (correlation)': 'float64',  
    'p-value (non-inferiority)': 'float64',
    "Cohen's d": 'float64'        
}

df = pd.DataFrame({col: pd.Series(dtype=dtype) for col, dtype in column_specs.items()})

# Iterate over the combinations of model sizes and window sizes, read the corresponding results, and compute the statistics.

for model_size in model_sizes:
    for win_size in win_sizes:

        if win_size == 1024 and model_size == 'base':
            continue # Skip the combination of MOMENT-base and window size 1024, as it is not covered in the experiments.

        path_model = f'../results/tsfms/moment_{model_size}_{win_size}.csv'
        path_baseline = f'../results/baselines/var_{win_size}.csv'

        results_model = pd.read_csv(path_model)
        results_baseline = pd.read_csv(path_baseline)

        x = results_model[metric].to_numpy()
        y = results_baseline[metric].to_numpy()

        # One series in the TSB-AD-U evaluation set is shorter than 1024, which results in a None value for the metric when window size is 1024.
        # We remove this value for the statistical analysis.
        if win_size == 1024:
            mask = ~np.isnan(y)
            x = x[mask]
            y = y[mask]

        scores_a = x.astype(np.float32)
        scores_b = y.astype(np.float32)
        
        correlation, p_value_correlation, p_value_noninferior, cohens_d = get_statistics(scores_a, scores_b, margins[(model_size, metric)])

        new_row_data = pd.Series({'Variant': '\\textbf{' + f'MOMENT-{model_size}-{win_size}' + '}', 'mean (TSFM)':scores_a.mean(), 'mean (baseline)':scores_b.mean(), 'correlation': correlation, 'p-value (correlation)': p_value_correlation, 'p-value (non-inferiority)' : p_value_noninferior, "Cohen's d": cohens_d})
        new_row_df = new_row_data.to_frame().T
        df = pd.concat([df, new_row_df], ignore_index=True)

df

In [ ]:
# Format the DataFrame for LaTeX output.

format_dict = {col: '{:.3f}' for col in df.select_dtypes(include='object').columns}
format_dict['p-value (correlation)'] = '{:.2e}'
del format_dict['Variant']

print(df[['Variant', 'mean (TSFM)', 'mean (baseline)', 'correlation', 'p-value (correlation)', "Cohen's d"]].style.format(format_dict).hide().to_latex())